
# Variational Quantum Simulation (VQS) for Mobile Network Security

This notebook applies **Variational Quantum Simulation (VQS)** with PennyLane to two mobile-network security tasks:

- **RF anomaly scoring** for rogue gNB / fake SSB / tone jammers from PSD features.
- **RACH preamble detection** as a quantum-inspired matched filter using correlation features.

> Requirements: `pennylane`, `numpy`, `matplotlib`. Run cell-by-cell.


In [2]:
!pip install pennylane

In [3]:
# If needed, install PennyLane
import sys
try:
    import pennylane as qml
    from pennylane import numpy as np
except Exception as e:
    print("PennyLane not found; attempting to install...")
    # Uncomment below if running locally without PennyLane
    # !pip -q install pennylane
    import pennylane as qml
    from pennylane import numpy as np
print("PennyLane version:", qml.__version__)

PennyLane version: 0.42.3


In [4]:

from __future__ import annotations
import numpy as np
import pennylane as qml
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import Callable, Optional, Sequence, Tuple

# Plot helper (Matplotlib only; no styles/colors specified)
def plot_energy_trace(energies, title):
    plt.figure(figsize=(7,4))
    plt.plot(energies)
    plt.xlabel("Step")
    plt.ylabel("Energy")
    plt.title(title)
    plt.grid(True)
    plt.show()



## VQS Core (Imaginary-Time, McLachlan)

We implement imaginary-time VQS with the **quantum metric tensor** and an energy gradient.


In [5]:

try:
    from scipy.linalg import cho_factor, cho_solve
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False

def _safe_solve(A: np.ndarray, b: np.ndarray, ridge: float = 1e-10) -> np.ndarray:
    Areg = A + ridge * np.eye(A.shape[0])
    if SCIPY_AVAILABLE:
        c, lower = cho_factor(Areg, lower=True, overwrite_a=False, check_finite=False)
        return cho_solve((c, lower), b, check_finite=False)
    return np.linalg.solve(Areg, b)

def _rk4(rhs: Callable[[np.ndarray], np.ndarray], theta: np.ndarray, dt: float) -> np.ndarray:
    k1 = rhs(theta)
    k2 = rhs(theta + 0.5*dt*k1)
    k3 = rhs(theta + 0.5*dt*k2)
    k4 = rhs(theta + dt*k3)
    return theta + (dt/6.0)*(k1 + 2*k2 + 2*k3 + k4)

@dataclass
class VQSConfig:
    n_qubits: int
    dev_name: str = "default.qubit"
    shots: Optional[int] = None
    diff_method: str = "parameter-shift"
    metric_diag_only: bool = False
    ridge: float = 1e-10

class VQS:
    """Imaginary-time VQS: A(θ) θ̇ = C(θ), with C = -½ ∇E."""
    def __init__(
        self,
        cfg: VQSConfig,
        ansatz: Callable[[Sequence[float]], None],
        hamiltonian: qml.Hamiltonian,
        theta0: np.ndarray,
    ):
        self.cfg = cfg
        self.theta = theta0.astype(float)
        self.H = hamiltonian
        self.dev = qml.device(cfg.dev_name, wires=cfg.n_qubits, shots=cfg.shots)

        @qml.qnode(self.dev, diff_method=cfg.diff_method)
        def energy_qnode(params):
            ansatz(params)
            return qml.expval(self.H)
        self.energy_qnode = energy_qnode

        mt_kw = {} if not cfg.metric_diag_only else {"approx": True}
        self.metric_fn = qml.metric_tensor(self.energy_qnode, **mt_kw)
        self.gradE_fn = qml.grad(self.energy_qnode)

    def _A(self, th):
        A = self.metric_fn(th)
        return np.diag(A) if (self.cfg.metric_diag_only and A.ndim==1) else np.real(A)

    def _C(self, th):
        return -0.5*np.array(self.gradE_fn(th))

    def _step_euler(self, dt: float) -> Tuple[np.ndarray, float]:
        thetadot = _safe_solve(self._A(self.theta), self._C(self.theta), ridge=self.cfg.ridge)
        self.theta = self.theta + dt*thetadot
        return self.theta.copy(), float(self.energy_qnode(self.theta))

    def _step_rk4(self, dt: float) -> Tuple[np.ndarray, float]:
        def rhs(th):
            return _safe_solve(self._A(th), self._C(th), ridge=self.cfg.ridge)
        self.theta = _rk4(rhs, self.theta, dt)
        return self.theta.copy(), float(self.energy_qnode(self.theta))

    def run(self, steps: int, dt: float, method: str="rk4") -> Tuple[np.ndarray, np.ndarray]:
        th_hist = [self.theta.copy()]
        E_hist = [float(self.energy_qnode(self.theta))]
        for _ in range(steps):
            _, E = (self._step_rk4(dt) if method=="rk4" else self._step_euler(dt))
            th_hist.append(self.theta.copy()); E_hist.append(E)
        return np.stack(th_hist), np.array(E_hist)



## Encodings and Ansatz


In [6]:

def normalize(v: np.ndarray) -> np.ndarray:
    v = np.asarray(v, float)
    n = np.linalg.norm(v) + 1e-12
    return v / n

def hardware_efficient_ansatz(n_qubits: int, layers: int = 2) -> Callable[[Sequence[float]], None]:
    wires = list(range(n_qubits))
    expected = layers * n_qubits * 2
    def _ansatz(params):
        p = np.array(params)
        assert p.size == expected, f"Expected {expected} params, got {p.size}"
        p = p.reshape(layers, 2, n_qubits)
        for l in range(layers):
            for q in wires:
                qml.RY(p[l,0,q], wires=q); qml.RZ(p[l,1,q], wires=q)
            for q in wires:
                qml.CNOT(wires=[q, (q+1) % n_qubits])
    return _ansatz



## RF Anomaly Scoring from PSD Features

We convert a small set of PSD features into a diagonal Hamiltonian that penalizes deviations from a benign template. The final energy is an **anomaly score**.


In [8]:

def topk_indices(power: np.ndarray, k: int) -> np.ndarray:
    k = min(k, power.size)
    return np.argsort(power)[-k:]

def psd_to_hamiltonian(psd: np.ndarray, benign_template: np.ndarray, wires: Sequence[int]) -> qml.Hamiltonian:
    psd_n = normalize(psd)
    tmpl_n = normalize(benign_template)
    delta = np.clip(psd_n - tmpl_n, -1.0, 1.0)
    coeffs, ops = [], []
    for i, w in enumerate(delta[: len(wires)]):
        coeffs.append(abs(float(w)) + 1e-3)
        ops.append(qml.PauliZ(int(wires[i])))
    return qml.Hamiltonian(coeffs, ops)

def rf_anomaly_vqs(psd: np.ndarray, benign_template: np.ndarray, n_qubits: int = 6, layers: int = 2,
                   steps: int = 100, dt: float = 0.08, diag_metric: bool = True) -> Tuple[float, np.ndarray, np.ndarray]:
    wires = list(range(n_qubits))
    H = psd_to_hamiltonian(psd, benign_template, wires)
    ans = hardware_efficient_ansatz(n_qubits, layers)
    theta0 = 0.1*np.random.randn(layers * n_qubits * 2)
    cfg = VQSConfig(n_qubits=n_qubits, metric_diag_only=diag_metric, ridge=1e-9)
    vqs = VQS(cfg, ans, H, theta0)
    thetas, energies = vqs.run(steps=steps, dt=dt, method="rk4")
    anomaly_score = float(energies[-1])
    return anomaly_score, thetas, energies

def _periodogram(x, fs):
    X = np.fft.rfft(x * np.hanning(len(x)))
    Pxx = (np.abs(X)**2) / (fs * len(x))
    f = np.fft.rfftfreq(len(x), 1/fs)
    return f, Pxx



### Demo: Rogue gNB / Tone-Style Anomaly


In [9]:

np.random.seed(7)
fs = 30e3; N = 2**15; t = np.arange(N)/fs
carriers = [2e3, 10e3, 18e3]
s = sum(np.cos(2*np.pi*f*t) for f in carriers)
benign = s + np.random.normal(0, 0.8, size=N)
rogue = s + 0.18*np.cos(2*np.pi*12.6e3*t) + np.random.normal(0, 0.8, size=N)

f, P_ben = _periodogram(benign, fs)
_, P_att = _periodogram(rogue, fs)

K = 6
idx = topk_indices(P_ben, K)
psd_feat = P_att[idx]
tmpl_feat = P_ben[idx]

score, _, Ehist = rf_anomaly_vqs(psd_feat, tmpl_feat, n_qubits=K, layers=2, steps=80, dt=0.08)
print(f"RF anomaly score (lower=benign, higher=anomalous): {score:.6f}")
plot_energy_trace(Ehist, "VQS Energy Trace — RF Anomaly Score")


/usr/local/lib/python3.12/dist-packages/pennylane/gradients/metric_tensor.py:382: UserWarning: Attempted to compute the metric tensor of a tape with no trainable parameters. If this is unintended, please mark trainable parameters in accordance with the chosen auto differentiation framework, or via the 'tape.trainable_params' property.
  warnings.warn(


QuantumFunctionError: No trainable parameters.


## RACH Preamble Detection as Quantum Matched Filter

We use correlation magnitudes around an expected offset to build a **reward** Hamiltonian. Lower final energy indicates a stronger match.


In [10]:

def rach_hamiltonian_from_corr(corr_mag: np.ndarray, expected_peak_idx: int, wires: Sequence[int]) -> qml.Hamiltonian:
    L = min(len(wires), corr_mag.size)
    corr_n = normalize(corr_mag[:L])
    sigma = max(1.0, 0.15*L)
    idx = np.arange(L)
    kernel = np.exp(-0.5*((idx - expected_peak_idx)**2) / (sigma**2))
    w = normalize(kernel) * (1.0 + 2.0*corr_n)
    coeffs, ops = [], []
    for i in range(L):
        coeffs.append(-float(w[i]))
        ops.append(qml.PauliZ(int(wires[i])))
    return qml.Hamiltonian(coeffs, ops)

def rach_vqs(corr_mag: np.ndarray, expected_peak_idx: int, n_qubits: int = 6, layers: int = 2,
             steps: int = 80, dt: float = 0.08, diag_metric: bool = True) -> Tuple[float, np.ndarray, np.ndarray]:
    wires = list(range(n_qubits))
    H = rach_hamiltonian_from_corr(corr_mag, expected_peak_idx, wires)
    ans = hardware_efficient_ansatz(n_qubits, layers)
    theta0 = 0.1*np.random.randn(layers * n_qubits * 2)
    cfg = VQSConfig(n_qubits=n_qubits, metric_diag_only=diag_metric, ridge=1e-9)
    vqs = VQS(cfg, ans, H, theta0)
    thetas, energies = vqs.run(steps=steps, dt=dt, method="rk4")
    return float(energies[-1]), thetas, energies



### Demo: RACH Preamble Offset Match


In [11]:

np.random.seed(11)
L = 8
true_offset = 5
corr = np.exp(-0.5*((np.arange(L)-true_offset)/1.2)**2) + 0.15*np.random.rand(L)

match_E, _, Ehist = rach_vqs(corr, expected_peak_idx=true_offset, n_qubits=L, layers=2, steps=60, dt=0.07)
print(f"RACH match energy (lower=better match): {match_E:.6f}")
plot_energy_trace(Ehist, "VQS Energy Trace — RACH Match")


QuantumFunctionError: No trainable parameters.


References

  *   McArdle, S., Endo, S., Aspuru-Guzik, A., Benjamin, S.C., & Yuan, X. (2019). Variational quantum simulation of imaginary time evolution. npj Quantum Information, 5, 75.
  *   Yuan, X. et al. (2019). Theory of variational quantum simulation. Quantum, 3, 191.
  *   Degen, C.L., Reinhard, F., & Cappellaro, P. (2017). Quantum Sensing. Reviews of Modern Physics, 89, 035002.
  *   Pirandola, S. et al. (2020). Advances in quantum cryptography. IEEE Journal on Selected Areas in Communications, 38(3), 193–246.
  *   Xu, W., Trappe, W., Zhang, Y., & Wood, T. (2021). The feasibility of launching and detecting jamming attacks in wireless networks. ACM MobiHoc.
  *   Sridharan, A., Sichitiu, M., & Brown, T. (2020). Security of timing protocols in mobile networks. IEEE Communications Magazine, 58(3), 45–51.
  *   3GPP TS 38.211 (2021). NR; Physical channels and modulation. 3rd Generation Partnership Project.
